# Capstone: Spin-chain simulation with mitigation

**Quantum Computing Advanced · Module 6 · about 5 hours** · notebook version 2026-10-09

**The question of this experiment.** How well can a noisy quantum computer follow the magnetization of a three-spin chain in time, how many Trotter steps should it use when every step adds noise, and how much do readout correction and zero-noise extrapolation recover?

You run the experiment on a **noisy simulator**, a simulator that adds the errors of a real quantum computer described by a **noise model**. You compare what you measure with what the noise model predicts, say how certain every result is, and say what the experiment cannot tell you. The capstone brings together the code, the mitigation and the application of Modules 2, 3 and 4, and you write more of the code yourself than in a lab.

**The plan.**

1. write the statistics functions every capstone uses (Step 1) and meet the capstone noise model (Step 2);
2. build the **Trotter circuit** of the transverse-field Ising chain and measure the **magnetization** (Step 3);
3. find the **trade-off** between Trotter error and gate noise as the number of steps grows (Step 4);
4. correct the **readout** error of the magnetization (Step 5);
5. fold the two-qubit gates and **extrapolate** to zero noise (Step 6);
6. follow M(t) from t = 0.25 to 3 with error bars (Step 7);
7. get your verification value (Step 8).

**How it is graded.** Your capstone quiz in Canvas (the one for this experiment, 20 points, pass at 16) asks for your verification value from the check cell (6 points), three results from this notebook and four questions about uncertainty, interpretation and limits (2 points each). Question 1 gives you three personal numbers: STEPS, P2 and READOUT. At the end you write a short **summary** of your experiment (required, not graded), which you submit in the Course Completion module.

**Saving your work.** JupyterLite keeps your changes in this browser, so you can stop and come back. To keep a copy elsewhere, use **File → Download**. Run each code cell with **Shift + Enter**, in order; if you come back later, run the cells from the top again.

## Step 0: start Python

Run the cell below. The first time, Python can take up to a minute to start in your browser, and Matplotlib takes a few more seconds to load.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import qsim
from qsim import QuantumCircuit, NoiseModel, ReadoutError, depolarizing_error, simulate_density_matrix

print("Ready. qsim", qsim.__version__, "| NumPy", np.__version__)

## Step 1: how certain is a result from shots?

Every number you measure in this capstone comes from shots, so every number has an uncertainty. Three formulas cover all of them.

**A fraction.** If a result has probability p and you take S shots, the fraction f of shots with that result has standard deviation

**σ = √( p (1 − p) / S )**.

**An expectation value of ±1 results**, such as ⟨Z⟩ or ⟨X X X⟩: each shot gives +1 or −1, and the average E has standard deviation

**σ = √( (1 − E²) / S )**.

This is twice the σ of the fraction of +1 results, because E = 2 f − 1. For E = 0.8 and S = 4,000 it is √(0.36 / 4000) = 0.0095.

**A combination of independent results.** Mitigation combines several measured values: an estimate Ê = c₁ E₁ + c₂ E₂ + ... from independent runs with uncertainties σ₁, σ₂, ... has

**σ = √( (c₁ σ₁)² + (c₂ σ₂)² + ... )**.

Uncertainties add **in quadrature**. The difference of two results is the special case c = (1, −1): σ_diff = √(σ₁² + σ₂²). Large coefficients, as in an extrapolation, make the combination much less certain than any single run: this is the price of mitigation you met in Module 3.

**The course rule: 3σ.** A measured value agrees with an expected value when they differ by at most 3σ. A difference larger than 3σ_diff is **significant**: shot noise alone would almost never produce it. A smaller difference may be real, but your data cannot show it. Two cautions: the rule treats the spread as normal (bell-shaped), which is good when many shots give each result; and when you compare many settings, chance alone makes one "significant" difference likely somewhere (with 20 comparisons, about 5%). Treat a single surprising difference among many as a question to test again with new shots.

**Your task:** write the three functions below. Every later step uses them.

- `fraction_sigma(p, shots)`: √(p (1 − p) / shots).
- `expectation_sigma(E, shots)`: √((1 − E²) / shots).
- `combined_sigma(coefficients, sigmas)`: √(Σ (cᵢ σᵢ)²) for two lists of the same length.

In [ ]:
def fraction_sigma(p, shots):
    """The standard deviation of a fraction measured with `shots` shots, when the true probability is p."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete fraction_sigma() first.")


def expectation_sigma(E, shots):
    """The standard deviation of the average of `shots` results of +1 or -1 with expectation value E."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete expectation_sigma() first.")


def combined_sigma(coefficients, sigmas):
    """The standard deviation of sum(c_i * E_i) for independent E_i with standard deviations sigma_i."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete combined_sigma() first.")

In [ ]:
print("fraction_sigma(0.0613, 4000) =", round(fraction_sigma(0.0613, 4000), 5), "  (expected 0.00379)")
print("expectation_sigma(0.8, 4000) =", round(expectation_sigma(0.8, 4000), 5), "  (expected 0.00949)")
print("combined_sigma([1, -1], [0.01, 0.02]) =", round(combined_sigma([1, -1], [0.01, 0.02]), 5), "  (expected 0.02236)")
print("combined_sigma([1.875, -1.25, 0.375], [0.012, 0.015, 0.02]) =",
      round(combined_sigma([1.875, -1.25, 0.375], [0.012, 0.015, 0.02]), 5), "  (expected 0.03023)")

## Step 2: the capstone noise model

The noise model of the capstone is the one you used in Level 2 Module 6, with one addition:

- after every **two-qubit gate**, a **depolarizing error** with parameter P2 (with probability P2 the two qubits are replaced by a completely random state);
- after every **one-qubit gate**, the same on one qubit with P2 / 10;
- an **idle error** on `id` gates, used only by the repetition-code capstone (a qubit that waits while others are measured);
- at **readout**, a 0 is recorded as 1 with probability READOUT and a 1 as 0 with probability 2 · READOUT.

The course values are **P2 = 0.01** and **READOUT = 0.02**, several times worse than IBM's best devices, so that the effects show clearly in a few thousand shots. Your capstone quiz gives you your own P2 and READOUT (0.005 to 0.030 each) for the check cell.

The prepared cell builds the model and three tools:

- `exact_probabilities(qc, p2, readout)`: the noise model's **exact** probability of every recorded result, from the density matrix (Module 1) and the readout confusion; no shots, no randomness. This is the expected value the 3σ rule compares with.
- `sample_counts(probs, shots, seed)`: random shots drawn from those probabilities. A run of the noisy simulator gives counts with exactly these statistics; drawing them from the exact probabilities is much faster in the browser, and the seed makes your run repeatable.
- `apply_readout(probs, readout)`: the readout errors alone, applied to probabilities.

In [ ]:
ONE_QUBIT_GATES = ["h", "x", "y", "z", "rx", "ry", "rz", "p", "s", "sdg", "t", "tdg", "sx"]
TWO_QUBIT_GATES = ["cx", "cz", "cp", "rzz", "swap"]
P2_COURSE, READOUT_COURSE = 0.01, 0.02


def confusion(readout):
    """Prepared. Rows: the true bit (0, 1); columns: the recorded bit."""
    return np.array([[1 - readout, readout], [2 * readout, 1 - 2 * readout]])


def capstone_noise_model(p2=P2_COURSE, readout=READOUT_COURSE, idle=0.0):
    """Prepared: depolarizing error p2 after two-qubit gates, p2/10 after one-qubit gates, idle on id gates, readout."""
    nm = NoiseModel()
    if p2 > 0:
        nm.add_all_qubit_quantum_error(depolarizing_error(p2 / 10, 1), ONE_QUBIT_GATES)
        nm.add_all_qubit_quantum_error(depolarizing_error(p2, 2), TWO_QUBIT_GATES)
    if idle > 0:
        nm.add_all_qubit_quantum_error(depolarizing_error(idle, 1), ["id"])
    if readout > 0:
        nm.add_all_qubit_readout_error(ReadoutError(confusion(readout).tolist()))
    return nm


def apply_readout(probs, readout):
    """Prepared. probs over m recorded bits (index bit j = classical bit j); each bit is flipped by the confusion."""
    probs = np.asarray(probs, dtype=float)
    m = int(round(math.log2(len(probs))))
    c = confusion(readout)
    for j in range(m):
        t = probs.reshape(2 ** (m - 1 - j), 2, 2 ** j)
        probs = np.einsum("aib,ij->ajb", t, c).reshape(-1)
    return probs


def exact_probabilities(qc, p2=P2_COURSE, readout=READOUT_COURSE, idle=0.0):
    """Prepared: the noise model's exact probability of each recorded result, {"bits": p}, Qiskit's key order.
    qc must end with its measurements (each classical bit receives one qubit). No shots are taken."""
    measured = {inst.clbits[0]: inst.qubits[0] for inst in qc.data if inst.name == "measure"}
    body = qc.copy()
    body.remove_final_measurements()
    nm = capstone_noise_model(p2, 0.0, idle)
    rho = simulate_density_matrix(body, nm if not nm.is_ideal() else None)   # the exact mixed state after the noisy gates
    m = len(measured)
    probs = rho.probabilities(qargs=[measured[j] for j in range(m)])
    probs = apply_readout(probs, readout)                                       # then the readout errors
    return {format(i, f"0{m}b"): float(p) for i, p in enumerate(probs) if p > 1e-15}


def sample_counts(probs, shots, seed):
    """Prepared: `shots` random results drawn from exact probabilities, as a quantum computer gives them."""
    keys = sorted(probs)
    p = np.array([probs[k] for k in keys])
    draw = np.random.default_rng(seed).multinomial(shots, p / p.sum())
    return {k: int(c) for k, c in zip(keys, draw) if c > 0}


print(capstone_noise_model())

## Step 3: the spin chain and its magnetization

The model is the one of Module 4 Track B: an open chain of N = 3 spins with the Hamiltonian

H = −J Σ Zᵢ Zᵢ₊₁ − h Σ Xᵢ, with J = 1 and h = 0.7.

All spins start up, |000⟩, and the quantity you follow is the **magnetization** M(t) = (1/N) Σ ⟨Zᵢ⟩, which starts at 1. Its exact value at T = 2 is **0.3862** (from the matrix exponential of H, computed in the next cells).

A quantum computer cannot apply e^(−iHT) in one gate. The **first-order Trotter** circuit splits T into `steps` steps of length dt = T / steps, and in each step applies the two parts of H one after the other:

- e^(+i dt Zᵢ Zᵢ₊₁) on every neighbouring pair: **RZZ(−2 dt)**, because RZZ(θ) = e^(−iθ Z Z / 2);
- e^(+i h dt Xᵢ) on every spin: **RX(−2 h dt)**.

The two parts do not commute, so the circuit is only approximately right: the **Trotter error** shrinks as dt shrinks.

From shots, each qubit's ⟨Zᵢ⟩ is (fraction of 0s) − (fraction of 1s). Qubit i is character N − 1 − i of a key, but M averages over all qubits, so the order does not matter here.

**Your task:**

- `trotter_circuit(N, h, T, steps)`: the circuit on N qubits; in every step, all the RZZ gates first (`qc.rzz(theta, i, i + 1)`), then the RX gates (`qc.rx(theta, i)`). **No measurements**: Step 6 folds the circuit.
- `magnetization(probs)`: (1/N) Σᵢ ⟨Zᵢ⟩ from counts or probabilities; divide by the total weight.

The prepared `magnetization_sigma(probs, shots)` gives the σ of M from shots. M is not a ±1 average (each shot gives one of 1, 1/3, −1/3, −1), so `expectation_sigma` does not apply: σ = √((⟨m²⟩ − M²) / shots), where m is the value of one shot.

In [ ]:
def trotter_circuit(N, h, T, steps):
    """`steps` first-order Trotter steps of dt = T / steps: RZZ(-2 dt) on each pair, then RX(-2 h dt) on each qubit."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete trotter_circuit() first.")


def magnetization(probs):
    """(1/N) sum_i <Z_i> from counts or probabilities."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete magnetization() first.")

In [ ]:
N, H_FIELD, T = 3, 0.7, 2.0


def measured(qc):
    """Prepared: a copy of qc with every qubit measured (qubit i into classical bit i)."""
    out = QuantumCircuit(qc.num_qubits, qc.num_qubits)
    out.compose(qc, inplace=True)
    for i in range(qc.num_qubits):
        out.measure(i, i)
    return out


def magnetization_sigma(probs, shots):
    """Prepared: the standard deviation of M measured with `shots` shots, from the probabilities of each result."""
    total = sum(probs.values())
    n = len(next(iter(probs)))
    m = {k: sum(1 - 2 * int(b) for b in k) / n for k in probs}
    M = sum(v * m[k] for k, v in probs.items()) / total
    m2 = sum(v * m[k] ** 2 for k, v in probs.items()) / total
    return math.sqrt(max(m2 - M * M, 0.0) / shots)


def exact_magnetization(N, h, t):
    """Prepared: the exact M(t) from the matrix exponential of H (no Trotter error, no noise)."""
    X, Z, I2 = np.array([[0, 1], [1, 0]]), np.diag([1.0, -1.0]), np.eye(2)

    def op(single, i):
        m = np.array([[1.0]])
        for q in reversed(range(N)):
            m = np.kron(m, single if q == i else I2)
        return m
    H = -sum(op(Z, i) @ op(Z, i + 1) for i in range(N - 1)) - h * sum(op(X, i) for i in range(N))
    w, v = np.linalg.eigh(H)
    psi0 = np.zeros(2 ** N); psi0[0] = 1
    psi = v @ (np.exp(-1j * w * t) * (v.conj().T @ psi0))
    return float(np.real(np.vdot(psi, (sum(op(Z, i) for i in range(N)) / N) @ psi)))


print(trotter_circuit(N, H_FIELD, T, 2).draw())
M_EXACT = exact_magnetization(N, H_FIELD, T)
print("Exact M(2.0) =", round(M_EXACT, 4))
for steps in [2, 4, 6, 8, 10, 12]:
    ideal = magnetization(exact_probabilities(measured(trotter_circuit(N, H_FIELD, T, steps)), 0, 0))
    print(f"   {steps:2d} Trotter steps, no noise: M = {ideal:.4f}   Trotter error {ideal - M_EXACT:+.4f}")

**What to notice.** With 2 steps (dt = 1) the Trotter circuit gives 0.2500, far from 0.3862; with 8 steps it gives 0.3819, and the error keeps shrinking roughly as dt. Without noise, more steps is always better.

## Step 4: more steps, more noise

Every step adds N − 1 = 2 RZZ gates (each a two-qubit gate with error P2) and N = 3 RX gates. So more steps means a smaller Trotter error but more gate noise. Run the cell: for 2 to 12 steps it gives the noiseless Trotter value, the exact noisy value, and a run of 4,000 shots with its σ.

In [ ]:
print(f"{'steps':>6s}{'RZZ':>5s}{'no noise':>10s}{'noisy, exact':>14s}{'your run':>18s}{'error of noisy':>16s}")
raw_exact, probs_by_steps = {}, {}
for steps in [2, 4, 6, 8, 10, 12]:
    qc = trotter_circuit(N, H_FIELD, T, steps)
    ideal = magnetization(exact_probabilities(measured(qc), 0, 0))
    probs_by_steps[steps] = exact_probabilities(measured(qc))
    raw_exact[steps] = magnetization(probs_by_steps[steps])
    run = magnetization(sample_counts(probs_by_steps[steps], 4000, seed=steps))
    sg = magnetization_sigma(probs_by_steps[steps], 4000)
    print(f"{steps:6d}{2 * steps:5d}{ideal:10.4f}{raw_exact[steps]:14.4f}{run:11.4f} +- {sg:.4f}{raw_exact[steps] - M_EXACT:+16.4f}")
print("Exact M(2.0) =", round(M_EXACT, 4))

**What to notice.** The noisy value first gets closer to the exact 0.3862 as the Trotter error shrinks, then falls away as the gate noise builds up: the raw value is best at about **6 steps** (0.3393), and at 8 steps it is **0.3311**. This trade-off decides the number of steps on every real device: the best dt is set by the noise, not by the mathematics. Noise pulls every ⟨Zᵢ⟩ toward 0, so it pulls M toward 0, here downward. With σ ≈ 0.012 for 4,000 shots, neighbouring step numbers are hard to tell apart from a single run.

## Step 5: correct the readout

For one qubit, the capstone readout records a 0 as 1 with probability r₀ = READOUT and a 1 as 0 with r₁ = 2 · READOUT. If p₀ is the true probability of 0 and z = p₀ − p₁ the true ⟨Z⟩, the measured value is

z_measured = (1 − r₀ − r₁) · z + (r₁ − r₀).

The factor (1 − r₀ − r₁) shrinks ⟨Z⟩; the offset (r₁ − r₀) appears because 1s are misread more often than 0s, which pushes ⟨Z⟩ up. M is an average of ⟨Zᵢ⟩, so the same relation holds for M, and you can **invert it**:

z = (z_measured − (r₁ − r₀)) / (1 − r₀ − r₁).

This is the readout mitigation of the mitigation capstone, done on the expectation value instead of on the probabilities. It is exact for this readout model because M needs only one qubit at a time. It also scales the uncertainty: σ_corrected = σ / (1 − r₀ − r₁).

**Your task:** `correct_readout_z(z, readout)`: the corrected value.

In [ ]:
def correct_readout_z(z, readout):
    """<Z> (or M) corrected for readout errors r0 = readout (0 read as 1) and r1 = 2 readout (1 read as 0)."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete correct_readout_z() first.")

In [ ]:
for steps in [2, 8, 12]:
    print(f"{steps:2d} steps: raw {raw_exact[steps]:.4f}  ->  readout-corrected {correct_readout_z(raw_exact[steps], READOUT_COURSE):.4f}")
print("Fixed point: correct_readout_z(1/3, 0.02) =", round(correct_readout_z(1 / 3, 0.02), 4))
print("At t = 0 (M = 1, no gates): measured", round(magnetization(exact_probabilities(measured(QuantumCircuit(3)))), 4),
      " corrected", round(correct_readout_z(magnetization(exact_probabilities(measured(QuantumCircuit(3)))), 0.02), 4))

**What to notice.** Here the readout correction changes almost nothing (8 steps: 0.3311 to 0.3310). That is not a mistake. The readout maps z to r₀ + (1 − 3r₀) z, which leaves z = 1/3 unchanged, and M(2.0) happens to be close to 1/3. At t = 0, where M = 1, the readout error is large (0.04) and the correction removes it completely. A correction that does nothing at one point can still matter elsewhere: Step 7 shows it over the whole time range. The remaining error at T = 2 comes from the gates.

## Step 6: fold the two-qubit gates and extrapolate

The gate noise here comes mostly from the RZZ gates (error P2, against P2/10 for each RX). **Local folding** amplifies only them: every RZZ(θ) becomes

RZZ(θ), then (scale − 1)/2 pairs RZZ(−θ), RZZ(θ).

Ideally RZZ(−θ) RZZ(θ) does nothing, so the circuit is unchanged, but each RZZ makes its error, so scale 3 has three times the RZZ noise. Then **extrapolate**: fit a straight line through (scale, corrected M) at scales 1, 3, 5 by least squares and read it at scale 0.

**Your tasks:**

- `fold_rzz(qc, scale)`: a new circuit (`out = qc.copy_empty_like()`), going through `qc.data`: for an instruction with `inst.name == "rzz"`, take `th = float(inst.params[0])` and the qubits `a, b = inst.qubits`, add `out.rzz(th, a, b)` and the pairs; any other instruction is appended unchanged (`out.data.append(inst)`).
- `extrapolate_linear(scales, values)`: the intercept of the least-squares line, for example `np.polyfit(scales, values, 1)[1]`.

In [ ]:
def fold_rzz(qc, scale):
    """Every RZZ(theta) followed by (scale - 1) / 2 pairs RZZ(-theta), RZZ(theta); other gates unchanged."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete fold_rzz() first.")


def extrapolate_linear(scales, values):
    """The least-squares straight line through (scale, value), read at scale 0."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete extrapolate_linear() first.")

In [ ]:
SCALES, LIN = [1, 3, 5], [13 / 12, 1 / 3, -5 / 12]     # the linear-extrapolation weights of the mitigation capstone


def mitigated_run(steps, t, shots=4000, seed=0):
    """Corrected and folded: the exact mitigated M, and one run of `shots` shots per scale with its sigma."""
    base = trotter_circuit(N, H_FIELD, t, steps)
    ex = {s: exact_probabilities(measured(fold_rzz(base, s))) for s in SCALES}
    corr = {s: correct_readout_z(magnetization(ex[s]), READOUT_COURSE) for s in SCALES}
    run = {s: correct_readout_z(magnetization(sample_counts(ex[s], shots, seed + s)), READOUT_COURSE) for s in SCALES}
    sig = [magnetization_sigma(ex[s], shots) / (1 - 3 * READOUT_COURSE) for s in SCALES]
    return (extrapolate_linear(SCALES, [corr[s] for s in SCALES]), extrapolate_linear(SCALES, [run[s] for s in SCALES]),
            combined_sigma(LIN, sig), corr)


print(f"{'steps':>6s}{'corrected, scales 1, 3, 5':>28s}{'mitigated, exact':>18s}{'your run':>18s}{'error':>9s}")
mit = {}
for steps in [2, 4, 6, 8, 10, 12]:
    m_exact, m_run, sg, corr = mitigated_run(steps, T, seed=100 * steps)
    mit[steps] = m_exact
    vals = ", ".join(f"{corr[s]:.3f}" for s in SCALES)
    print(f"{steps:6d}{vals:>28s}{m_exact:18.4f}{m_run:11.4f} +- {sg:.4f}{m_exact - M_EXACT:+9.4f}")
print("Exact M(2.0) =", round(M_EXACT, 4), "  best number of steps after mitigation:", min(mit, key=lambda s: abs(mit[s] - M_EXACT)))

**What to notice.** Mitigation moves the 8-step value from 0.3310 to **0.3618**, and the best number of steps rises to **6** (0.3652): once the noise is partly removed, more steps can be afforded. What remains (about 0.021 at 6 steps) is the Trotter error, the RX noise that was not folded, and the curvature that a straight line misses. The price is in σ: about 0.015 for the mitigated value against 0.012 for the raw one, with three times the shots. Compare the error column with σ: from 4 to 10 steps the exact mitigated values differ by less than one σ, so a single run could not pick the best one; the exact noise model can.

## Step 7: the magnetization in time

A simulation is usually wanted at many times. Run the cell: with a fixed dt = 0.25 (so t / 0.25 steps), it computes M(t) for t = 0.25 to 3: exact, raw (4,000 shots), readout-corrected, and fully mitigated (4,000 shots per scale), with ±σ error bars.

In [ ]:
times = [0.25 * k for k in range(1, 13)]
rows = []
for k, t in enumerate(times, start=1):
    qc = trotter_circuit(N, H_FIELD, t, k)
    ex1 = exact_probabilities(measured(qc))
    raw_run = magnetization(sample_counts(ex1, 4000, seed=500 + k))
    raw_sig = magnetization_sigma(ex1, 4000)
    m_exact, m_run, m_sig, _ = mitigated_run(k, t, seed=600 + 10 * k)
    rows.append((t, exact_magnetization(N, H_FIELD, t), raw_run, raw_sig, correct_readout_z(raw_run, READOUT_COURSE), m_run, m_sig))
    print(f"t = {t:4.2f}: exact {rows[-1][1]:.4f}   raw {raw_run:.4f} +- {raw_sig:.4f}   corrected {rows[-1][4]:.4f}"
          f"   mitigated {m_run:.4f} +- {m_sig:.4f}")

fine = np.linspace(0, 3, 121)
plt.figure(figsize=(6.5, 3.8))
plt.plot(fine, [exact_magnetization(N, H_FIELD, t) for t in fine], "k-", lw=1, label="exact")
r = np.array(rows)
plt.errorbar(r[:, 0], r[:, 2], yerr=r[:, 3], fmt="s", ms=4, capsize=2, label="raw")
plt.errorbar(r[:, 0] + 0.03, r[:, 5], yerr=r[:, 6], fmt="o", ms=4, capsize=2, label="corrected + ZNE")
plt.xlabel("t"); plt.ylabel("M(t)"); plt.legend(); plt.title("3 spins, h = 0.7, dt = 0.25, 4,000 shots per circuit")
plt.tight_layout(); plt.show()

within = sum(abs(row[5] - row[1]) <= 3 * row[6] for row in rows)
print(f"Mitigated points within 3 sigma of the exact curve: {within} of {len(rows)}")

**What to notice.** The raw points sit below the exact curve at every time, and the gap grows with t, because longer times need more steps and more gates. At early times much of the gap is readout (M is close to 1), and the readout correction removes it; later the gate noise dominates. The mitigated points are much closer to the curve, with error bars about 1.3 times the raw ones, but not on it: even without shot noise, the mitigated values lie 0.004 to 0.028 below the exact curve, up to about two σ of 4,000 shots. So in a run, a few mitigated points may miss the curve by more than 3σ. A bias of one or two σ shows clearly only when many points miss in the same direction, as they do here, or when you take more shots.

The dip of M around t ≈ 1.6 and the rise after it are the physics: the transverse field turns the spins away from Z, and the coupling brings them partly back. A noisy device that only showed the raw points would get the shape right and the size wrong.

## Step 8: your personal check

Open your capstone quiz in Canvas. Question 1 shows your own numbers: STEPS (2 to 12), P2 and READOUT (0.005 to 0.030). Type them below and run the next two cells. The check cell tests the three statistics functions of Step 1 and every function you wrote for this capstone. Only if every test passes does it print your **verification value**: 10,000 times the exact mitigated magnetization at T = 2 (N = 3, h = 0.7) for your number of Trotter steps and your noise model: readout-corrected, RZZ gates folded at scales 1, 3 and 5, extrapolated linearly, rounded to a whole number. For example, STEPS, P2, READOUT = 8, 0.01, 0.02 gives 3618.

In [ ]:
STEPS = 0    # your number from Canvas
P2 = 0    # your number from Canvas
READOUT = 0    # your number from Canvas

In [ ]:
# The course's check code is in l3_m6_common.py and l3_m6_spin_check.py, next to this notebook.
# It is kept out of the notebook so that the course's reference solutions are not on screen.
# You may open the file to see how your functions are tested, but write your own functions first.
from l3_m6_spin_check import check_l3_m6_spin

passed, messages, value = check_l3_m6_spin(fraction_sigma, expectation_sigma, combined_sigma, trotter_circuit, magnetization,
                                           correct_readout_z, fold_rzz, extrapolate_linear, STEPS, P2, READOUT)
for m in messages:
    print(m)
if passed:
    print("\nAll tests passed. Your verification value is", value)
else:
    print("\nNot yet: fix the item above and run this cell again.")

## Limits of this experiment

- **Three spins.** A three-spin chain is easy for a classical computer, which is why you can compare with the exact answer. A useful quantum simulation needs a chain too large for that, and then the remaining bias can no longer be seen directly.
- **Only the RZZ gates were folded.** The RX gates and the readout are not amplified, so their part of the error is not extrapolated away. On hardware, RZZ is itself made of two CX gates (or one native two-qubit gate) and single-qubit gates, and the compiler may merge or cancel folded gates.
- **The noise model is simple.** Depolarizing errors and independent readout errors. Real devices also have coherent errors, which can add up step after step in a Trotter circuit, and crosstalk between neighbouring qubits.
- **First-order Trotter.** A second-order formula (half a step of one part, a full step of the other, half a step of the first) has a much smaller Trotter error for the same number of two-qubit gates, so it would favour fewer steps.
- **One quantity.** M averages three ⟨Zᵢ⟩. Correlations such as ⟨Zᵢ Zⱼ⟩ are more sensitive to noise and need more shots for the same σ.

These limits are not failures of your experiment: every experiment has them. Saying what a result does and does not show is part of reporting it.

## Optional: run it on a real IBM quantum computer

If you have an IBM Quantum account with an Open Plan instance (see **Set Up Your Tools** in Start Here), the [Module 6 hardware notebook in Colab](https://colab.research.google.com/github/CSU-Physics/quantum-computing-notebooks/blob/main/colab/level3/QC-L3-M6-hardware-colab.ipynb) runs this capstone's reference circuits on a real computer and compares them with the ideal values and with this noise model. Your API key stays in Colab's **Secrets**; never type it into a cell. The Open Plan gives up to 10 minutes of quantum computer time every 28 days; the hardware notebook uses well under a minute of it, but the queue can take from seconds to hours. This step is not graded, and the badge does not depend on it.

## Your capstone summary (required, not graded)

Write five short answers here (double-click this cell to edit it). They are how a result is reported in a lab notebook or a paper, and they are good preparation for the capstone quiz. Then paste them into **Your Capstone Summary** in the Course Completion module in Canvas: it is not graded, but it is required for the badge.

1. **Question.** What did you want to find out?
2. **Method.** Which circuits, which noise model (P2, READOUT), how many shots, which mitigation or decoding?
3. **Result.** Your main numbers, each with its uncertainty (for example 0.976 ± 0.010).
4. **Interpretation.** What do the numbers say about the question? Which differences are significant (more than 3σ)?
5. **Limits.** What does this experiment not show?

**Next in Canvas:** the **Capstone Quiz — Spin Chain**, then the Module 6 time log.